## part 01 :- Structured output

In [1]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
from pydantic import BaseModel, Field

In [2]:
load_dotenv(override=True)

True

In [3]:
import os
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

PROVIDER = "gemini"  # "gemini", "groq", හෝ "openai" තෝරන්න

if PROVIDER == "gemini":
    set_tracing_disabled(True)
    external_client = AsyncOpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.getenv("GEMINI_API_KEY"),
    )
    MODEL = OpenAIChatCompletionsModel(
        model="gemini-3.5-flash-lite",
        openai_client=external_client
    )

elif PROVIDER == "groq":
    set_tracing_disabled(True)
    external_client = AsyncOpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY"),
    )
    MODEL = OpenAIChatCompletionsModel(
        model="openai/gpt-oss-120b",
        openai_client=external_client
    )

else:
    # OpenAI පාවිච්චි කරනවා නම්
    MODEL = "gpt-4.1-mini"

In [4]:
instructions = """
You are a sales agent working for a hospital,
a healthcare institution that provides high-quality medical care,
specialist consultations, diagnostic services, and patient-focused
treatment solutions.

You write compelling, professional, and empathetic sales emails
that promote the hospital's healthcare services, build trust with
potential patients, and encourage them to make inquiries or book
appointments.
"""

In [5]:
email_writer = Agent(name="Email Writer", instructions=instructions, model=MODEL)
response = await Runner.run(email_writer, "Write a sales email promoting the hospital's healthcare services to potential patients.")

In [6]:
email = response.final_output
print(email)

Subject: Your Health and Well-being are Our Top Priority at [Hospital Name]

Dear [Patient Name],

In a fast-paced world, it’s easy to put our health on the back burner. But true wellness starts with proactive care, and having a trusted healthcare partner by your side makes all the difference. 

At [Hospital Name], we believe that exceptional healthcare is about more than just treating illnesses—it’s about nurturing your overall well-being. Whether you are looking for routine preventative care, advanced diagnostics, or specialized medical treatment, our dedicated team of world-class physicians, nurses, and specialists are here to provide the compassionate, personalized care you deserve.

**Why Choose [Hospital Name]?**

*   **Expert Specialists:** Access to leading professionals across cardiology, orthopedics, pediatrics, oncology, and more.
*   **State-of-the-Art Technology:** Advanced diagnostic and surgical tools ensuring accurate diagnoses and faster, safer recoveries.
*   **Patien

In [7]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the mail, not including the greeting and signature")
    contains_placeholder: bool = Field(description="Whether the email contains a placeholder for personalization")

In [8]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the mail, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholder': {'description': 'Whether the email contains a placeholder for personalization',
   'title': 'Contains Placeholder',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholder'],
 'title': 'EmailReview',
 'type': 'object'}

In [9]:
checker = Agent(name="Checker", instructions="You review potential emails", model=MODEL, output_type=EmailReview)
result = await Runner.run(checker, email)

In [10]:
review = result.final_output
review

EmailReview(is_professional=True, number_of_sentences=11, contains_placeholder=True)

In [11]:
review.is_professional

True

In [12]:
review.contains_placeholder

True

## Part 2 :- Guardrails

In [13]:
@output_guardrail
async def email_guardrails(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholder or not review.is_professional
    return GuardrailFunctionOutput(
        output_info={"review": review},
        tripwire_triggered=is_problem
    )

In [14]:
another_ins = instructions + "\n speak like a unprofessional and unempathetic sales agent, use placeholders for personalization, and write in a casual tone"

sales_agent = Agent(name="Sales Agent", instructions=another_ins, model=MODEL, output_guardrails=[email_guardrails])

In [15]:
result = await Runner.run(sales_agent, "write a cold sales email")

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

## Me pahala widiya thama ap practise wenna hodfama de

In [16]:
simple_cowboy = Agent(name="Simple Cowboy", instructions="You are a cowboy who speaks in a simple and straightforward manner.", model=MODEL)
result = await Runner.run(simple_cowboy, "write a cold sales email")
email = result.final_output
print(email)

Howdy,

I’ll keep this short and sweet, 'cause I know your time's valuable. 

I help folks like you get more work done without having to work harder. Back at the ranch, we call that working smarter, not harder. 

If you got five minutes this week, let’s have a quick chat and see if we can put some more coin in your pocket. 

Let me know when you're free, and we'll saddle up.

Best,

[Your Name]


In [17]:
result = await Runner.run(checker, email)
review = result.final_output
if review.is_professional and not review.contains_placeholder:
    print("The email is professional and does not contains placeholders.")
else:
    print("The email is either unprofessional or contains placegholders.")


The email is either unprofessional or contains placegholders.
